In [1]:
import pandas as pd

test_df = pd.read_csv("/Users/omkar/Documents/SmartShop/app/sains_food_cupboard.csv")

In [2]:
test_df.head()

,supermarket,prices_(£),prices_unit_(£),unit,names,date,category,own_brand
0,Sains,0.90,7.60,kg,Maryland Cookies Chocolate Chip Minis x6,2024-04-13,food_cupboard,False
1,Sains,3.00,0.13,unit,Weetabix Cereal x24,2024-04-13,food_cupboard,False
2,Sains,1.10,6.90,kg,Walker's Shortbread Fingers x10 160g,2024-04-13,food_cupboard,False
3,Sains,1.45,0.64,l,Sainsbury's British Semi Skimmed Milk 2.27L (4...,2024-04-13,food_cupboard,True
4,Sains,0.78,0.16,unit,Sainsbury's Fairtrade Bananas x5,2024-04-13,food_cupboard,True


In [3]:
df = test_df[['names', 'own_brand']].copy()

In [4]:
df

,names,own_brand
0,Maryland Cookies Chocolate Chip Minis x6,False
1,Weetabix Cereal x24,False
2,Walker's Shortbread Fingers x10 160g,False
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True
4,Sainsbury's Fairtrade Bananas x5,True
...,...,...
508709,Reese's Peanut Butter Crème Egg 5 x 34g (170g),False
508710,Sainsbury's Fig Rolls 200g,True
508711,Sainsbury's Conchiglie (Shells) 500g,True
508712,Paxo Veggie Fillers Tomato & Herb 120g,False


In [5]:
##### Function to normalize the text in the names column.

import re

def clean_product_name(text):
    text = text.lower()

    # Remove patterns like "2 x 500ml"
    text = re.sub(r'\d+\s*[xX]\s*\d*\.?\d+\s*(kg|g|l|ml)', '', text)

    # Remove single weights like "500g", "1.5l"
    text = re.sub(r'\d*\.?\d+\s*(kg|g|l|ml)', '', text)

    # Remove pack info like "x6", "pack of 6"
    text = re.sub(r'(x|pack of)\s*\d+', '', text)

    # Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    text = text.replace('(', '').replace(')', '')

    return text

In [6]:
df['cleaned_names'] = df['names'].apply(lambda x: clean_product_name(x))

In [7]:
df.head(3)

,names,own_brand,cleaned_names
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis
1,Weetabix Cereal x24,False,weetabix cereal
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers


In [8]:
#### Subset the data to only include 3rd party brand products.

third_party_df = df[df['own_brand'] == False].copy()

In [9]:
third_party_df.head()

,names,own_brand,cleaned_names
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis
1,Weetabix Cereal x24,False,weetabix cereal
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers
8,Stamford Street Co. Chopped Tomatoes in Tomato...,False,stamford street co. chopped tomatoes in tomato...
12,Cravendale Filtered Fresh Semi Skimmed Milk 2L...,False,cravendale filtered fresh semi skimmed milk fr...


In [10]:
####  Checking unique products and their counts.
third_party_df['cleaned_names'].value_counts().head(20)

cleaned_names
heinz baked beans in a rich tomato sauce                   539
heinz cream of tomato soup                                 364
heinz no added sugar baked beans in a rich tomato sauce    364
weetabix cereal                                            363
tilda pure basmati rice                                    349
heinz spaghetti hoops in tomato sauce                      330
nutella hazelnut & chocolate spread                        316
cadbury dairy milk chocolate bar                           303
heinz seriously good light mayonnaise                      276
lindt gold bunny milk chocolate                            276
walkers classic variety multipack crisps                   275
nature's finest peach in juice                             275
hellmann's real squeezy mayonnaise                         274
quaker oat so simple golden syrup porridge sachets         274
princes corned beef                                        274
lindt lindor milk chocolate truffles box 

In [25]:
#### Generating n-gramns:
from collections import Counter

def generate_ngrams(names, n):
    ngrams = []
    
    for name in names:
        tokens = name.split()
        if len(tokens) >= n:
            ngrams.append(" ".join(tokens[:n]))
    
    return ngrams

#Count frequency
names = df['cleaned_names'].dropna().str.lower()

one_word = Counter(generate_ngrams(names, 1))
two_word = Counter(generate_ngrams(names, 2))
three_word = Counter(generate_ngrams(names, 3))

In [26]:
one_word.most_common(20), two_word.most_common(20), three_word.most_common(20)

([("sainsbury's", 135189),
  ('heinz', 11721),
  ('cadbury', 10490),
  ('lindt', 5558),
  ('walkers', 5339),
  ("kellogg's", 5226),
  ('batchelors', 5059),
  ('twinings', 4567),
  ("mcvitie's", 4151),
  ('john', 4073),
  ('dr.', 4034),
  ('the', 3981),
  ("hartley's", 3799),
  ('quaker', 3549),
  ('baxters', 3276),
  ('old', 3243),
  ('stamford', 3188),
  ('princes', 3150),
  ("jacob's", 3107),
  ('fudco', 2890)],
 [('john west', 4073),
  ('dr. oetker', 3942),
  ('cadbury dairy', 3250),
  ('stamford street', 3188),
  ('old el', 3037),
  ('quaker oat', 2889),
  ("sainsbury's fairtrade", 2778),
  ('loyd grossman', 2297),
  ('bonne maman', 2007),
  ('tilda microwave', 1858),
  ("tony's chocolonely", 1800),
  ("sainsbury's deliciously", 1784),
  ("sainsbury's ground", 1763),
  ("sainsbury's sweet", 1730),
  ("sainsbury's tomato", 1682),
  ("sainsbury's ready", 1615),
  ('the spice', 1603),
  ('lindt lindor', 1585),
  ('heinz baked', 1579),
  ("sainsbury's italian", 1574)],
 [('cadbury dair

In [36]:
import pandas as pd

df1 = pd.DataFrame(one_word.items(), columns=["one", "one_score"])
df2 = pd.DataFrame(two_word.items(), columns=["two", "two_score"])
df3 = pd.DataFrame(three_word.items(), columns=["three", "three_score"])

,one,one_score
0,maryland,431
1,weetabix,1282
2,walker's,372
3,sainsbury's,135189
4,stamford,3188
...,...,...
938,scrambled,9
939,cidona,14
940,pulsin,6
941,j-basket,5


In [38]:
def get_prefix(word, n):
    return " ".join(word.split()[:n])

In [41]:
df2["one"] = df2["two"].apply(lambda x: get_prefix(x, 1))

merged_12 = df2.merge(df1, on="one", how="left")
merged_12 = merged_12.rename(columns={"one_score_x": "two_score"})

In [42]:
df3["two"] = df3["three"].apply(lambda x: get_prefix(x, 2))

merged_23 = df3.merge(df2, on="two", how="left")

In [51]:
final_df = merged_23.merge(
    merged_12[["one", "two", "one_score", "two_score"]],
    on="two",
    how="left"
)

In [52]:
final_df.sort_values(by="one_score", ascending=False)

,three,three_score,two,two_score_x,one_x,one_y,one_score,two_score_y
2559,sainsbury's mild beef,91,sainsbury's mild,563,sainsbury's,sainsbury's,135189,563
1039,"sainsbury's brioche loaf,",54,sainsbury's brioche,54,sainsbury's,sainsbury's,135189,54
1037,sainsbury's chicken noodle,84,sainsbury's chicken,1394,sainsbury's,sainsbury's,135189,1394
2279,sainsbury's aromatic korma,52,sainsbury's aromatic,143,sainsbury's,sainsbury's,135189,143
2282,sainsbury's italian slow,92,sainsbury's italian,1574,sainsbury's,sainsbury's,135189,1574
...,...,...,...,...,...,...,...,...
5092,spicy yuzu paste,6,spicy yuzu,6,spicy,spicy,6,6
5209,ben & jerry's,5,ben &,5,ben,ben,5,5
5278,rhythmluten free super,5,rhythmluten free,5,rhythmluten,rhythmluten,5,5
5274,j-basket seaweed crisps,5,j-basket seaweed,5,j-basket,j-basket,5,5


In [49]:
def check_chain(row):
    one = row["one_x"]
    two = row["two"]
    three = row["three"]
    
    if not all(isinstance(x, str) for x in [one, two, three]):
        return False
    
    return (
        two.startswith(one + " ") and
        three.startswith(two + " ") and
        len(two.split()) == len(one.split()) + 1 and
        len(three.split()) == len(two.split()) + 1
    )
final_df.apply(check_chain, axis=1)

0       True
1       True
2       True
3       True
4       True
        ... 
5290    True
5291    True
5292    True
5293    True
5294    True
Length: 5295, dtype: bool

In [ ]:
third_party_df[third_party_df['cleaned_names'].str.startswith('maryland')]['cleaned_names'].value_counts().head(20)

In [ ]:
#### SComputing scores to create a brand set.

from collections import defaultdict

prefix_next_map = {
    1: defaultdict(list),
    2: defaultdict(list),
    3: defaultdict(list)
}

names = third_party_df['cleaned_names'].dropna().str.lower()

for name in names:
    tokens = name.split()
    
    for n in [1, 2, 3]:
        if len(tokens) > n:
            prefix = " ".join(tokens[:n])
            next_token = tokens[n]
            
            prefix_next_map[n][prefix].append(next_token)


In [ ]:
import pandas as pd

def compute_scores(prefix_map):
    data = []
    
    for n in prefix_map:
        for prefix, next_tokens in prefix_map[n].items():
            freq = len(next_tokens)
            diversity = len(set(next_tokens))
            
            score = freq * (diversity + 1)  # simple scoring
            
            data.append({
                "prefix": prefix,
                "n": n,
                "freq": freq,
                "diversity": diversity,
                "score": score
            })
    
    return pd.DataFrame(data)

score_df = compute_scores(prefix_next_map)

In [ ]:
score_df.sort_values(by="score", ascending=False).head(20)

In [ ]:
score_df['prefix'].size

In [ ]:
score_df['percentile'] = score_df['score'].rank(pct=True)
score_df['percentile'] = score_df['percentile'] * 100

In [ ]:
score_df.sort_values(by="percentile", ascending=False).head(20)

In [ ]:
score_df['percentile_within_n'] = score_df.groupby('n')['score'].rank(pct=True)

In [ ]:
score_df.sort_values(by="percentile_within_n", ascending=False).info()

In [ ]:
### Extracting potential brands based on a percentile threshold.

# threshold = 0.9  # Example threshold, adjust as needed
# diversity_threshold = 20  # Example diversity threshold, adjust as needed

initial_brands = score_df[
    (
        (score_df['n'] == 1) & (score_df['percentile_within_n'] > 0.9)  & (score_df['diversity'] > 20)
    ) |
    (
        (score_df['n'] == 2) & (score_df['percentile_within_n'] > 0.85)
    ) |
    (
        (score_df['n'] == 3) & (score_df['percentile_within_n'] > 0.8)
    )
]['prefix'].tolist()

In [ ]:
initial_brands

In [ ]:
#### Cleaning the Brand Set

bad_words = {"in", "of", "to", "for", "and", "with","up",'cookies'}

brands_set = {b for b in set(initial_brands) if b not in bad_words and len(b) > 2}

In [ ]:
len(brands_set)

In [ ]:
### Extracting the list of prefix which has percentile more then 0.99
##brands_set_99 = set(score_df[score_df['percentile'] > 99]['prefix'].tolist())